# NB_patient_360
Builds `caresync_databricks.gold.patient_360` per STTM sheet **patient_360**.
* **Grain:** one row per patient
* **Driving table:** `silver.patients` (LEFT JOIN `hospitals`, `insurance_providers`, aggregated `lab_results`)
* **Filters:** none, all patients included
* **Load:** FULL, CREATE OR REPLACE TABLE ... AS SELECT
* **Data note:** `silver.insurance_providers` holds duplicate rows per `insurance_provider_id`. They are deduped to one row per id before the join so the one-row-per-patient grain holds.

In [0]:
%sql
CREATE OR REPLACE TABLE caresync_databricks.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary (STTM patient_360)'
AS
WITH insurance_dedup AS (
  -- silver.insurance_providers contains duplicate rows per id; keep one row per id
  SELECT insurance_provider_id, provider_name
  FROM caresync_databricks.silver.insurance_providers
  QUALIFY ROW_NUMBER() OVER (PARTITION BY insurance_provider_id ORDER BY updated_timestamp DESC, provider_name) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM caresync_databricks.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name,
  p.last_name,
  p.date_of_birth,
  CAST(FLOOR(datediff(current_date(), p.date_of_birth) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  ip.provider_name                                                       AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                               AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                       AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                       AS critical_test_count,
  la.last_test_date
FROM caresync_databricks.silver.patients p
LEFT JOIN caresync_databricks.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance_dedup ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [0]:
NB_patient_360_df = spark.table("caresync_databricks.gold.patient_360")
record_count_patient_360 = NB_patient_360_df.count()
dbutils.notebook.exit(str(record_count_patient_360))